# RAG Testing with Real SNOMED CT Data

This notebook tests the RAG feature against actual SNOMED UK Clinical RF2 data and MedCAT model pack.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from snomed_methods.rag import RAGChat, RAGExplanations, RAGRetriever

print("OK")

In [ ]:
MEDCAT_PATH = os.path.join(
    "..",
    "model_packs",
    "medcat_model_pack_422d1d38fc58f158.zip",
)

from medcat.cat import CAT

cat = CAT.load_model_pack(MEDCAT_PATH)
print(f"OK MedCAT loaded from {os.path.basename(MEDCAT_PATH)}")

In [ ]:
from snomed_methods.llm_concept_embedder import load_concepts_from_medcat

concept_df = load_concepts_from_medcat(cat)
print(f"OK Loaded {len(concept_df)} concepts from MedCAT")

keyword = "Meningioma"
meningioma_df = concept_df[
    concept_df["preferred_name"].str.contains(keyword, case=False, na=False)
]

print(f"Found {len(meningioma_df)} meningioma-related concepts")

In [ ]:
from snomed_methods.llm_concept_embedder import ClinicalConceptEmbedder

embedder = ClinicalConceptEmbedder(
    model_name_or_path="sentence-transformers/all-MiniLM-L6-v2",
    backend="hf",
    device="cpu",
)

concept_texts = embedder.prepare_concept_text(meningioma_df)
embeddings = embedder.generate_embeddings(concept_texts, batch_size=4)

cui_to_embedding = {}
for i, cui in enumerate(meningioma_df["cui"].tolist()):
    cui_to_embedding[cui] = embeddings[i]

cui_to_name = {row["cui"]: row["preferred_name"] for _, row in meningioma_df.iterrows()}

print(f"Generated {len(cui_to_embedding)} embeddings of dim {embeddings.shape[1]}")

In [ ]:
retriever = RAGRetriever(
    embedder=embedder,
    cui_to_embedding=cui_to_embedding,
    cui_to_name=cui_to_name,
)

print(f"FAISS index built with {len(retriever.cui_list)} concepts")

In [ ]:
results = retriever.retrieve("meningioma", top_k=15)

has_meningioma = any("Meningioma" in name for _, name, _ in results)

print(f"Retrieved {len(results)} concepts")
for i, (cui, name, score) in enumerate(results[:5], 1):
    print(f"{i}. [{score:.4f}] {name}")

print(f"Meningioma found: {has_meningioma}")

In [ ]:
explanations = RAGExplanations(embedder=embedder, backend="hf")
rag_chat = RAGChat(retriever=retriever, explanations=explanations)

response1 = rag_chat.ask(query="brain tumor meningioma", top_k=5)
print(f"Turn 1: {len(response1['results'])} results")

response2 = rag_chat.follow_up(feedback="atypical only", refine_with_previous=True)
print(f"Turn 2: {len(response2['results'])} results (context-aware)")

In [ ]:
checks = [
    ("FAISS Index", retriever.index is not None),
    ("Retrieval Works", len(results) > 0),
    ("Meningioma Found", has_meningioma),
]

print("Validation Results:")
for name, passed in checks:
    print(f"{'OK' if passed else 'FAIL'} {name}: {passed}")